In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("--- MEMBANGUN ARSITEKTUR SEMUA MODEL UNTUK PRESENTASI ---")

# =========================================================
# 1. DEEPCONN (Deep Cooperative Neural Networks)
# =========================================================
class DeepCoNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, pretrained_matrix, num_filters=100, kernel_size=3, latent_dim=32):
        super(DeepCoNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_matrix))
        self.embedding.weight.requires_grad = True
        
        self.user_conv = nn.Conv1d(embed_dim, num_filters, kernel_size)
        self.user_fc = nn.Linear(num_filters, latent_dim)
        
        self.item_conv = nn.Conv1d(embed_dim, num_filters, kernel_size)
        self.item_fc = nn.Linear(num_filters, latent_dim)
        
        self.dropout = nn.Dropout(0.2)
        self.prediction = nn.Linear(latent_dim * 2, 1)

    def forward(self, u_seq, i_seq):
        u_emb = self.embedding(u_seq).permute(0, 2, 1) 
        i_emb = self.embedding(i_seq).permute(0, 2, 1)
        
        u_c = F.relu(self.user_conv(u_emb))
        u_pool = F.max_pool1d(u_c, u_c.shape[2]).squeeze(2)
        u_feat = F.relu(self.user_fc(u_pool))
        
        i_c = F.relu(self.item_conv(i_emb))
        i_pool = F.max_pool1d(i_c, i_c.shape[2]).squeeze(2)
        i_feat = F.relu(self.item_fc(i_pool))
        
        concat = torch.cat([u_feat, i_feat], dim=1)
        return self.prediction(self.dropout(concat)).squeeze()

# =========================================================
# 2. MATRIX FACTORIZATION (ID-Based)
# =========================================================
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64):
        super(MatrixFactorization, self).__init__()
        self.u_emb = nn.Embedding(num_users, embed_dim)
        self.i_emb = nn.Embedding(num_items, embed_dim)
        self.u_bias = nn.Embedding(num_users, 1)
        self.i_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

    def forward(self, u_idx, i_idx):
        u_e = self.u_emb(u_idx)
        i_e = self.i_emb(i_idx)
        dot = (u_e * i_e).sum(dim=1)
        u_b = self.u_bias(u_idx).squeeze()
        i_b = self.i_bias(i_idx).squeeze()
        return dot + u_b + i_b + self.global_bias

# =========================================================
# 3. NEURAL COLLABORATIVE FILTERING (ID-Based)
# =========================================================
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64):
        super(NCF, self).__init__()
        self.u_emb = nn.Embedding(num_users, embed_dim)
        self.i_emb = nn.Embedding(num_items, embed_dim)
        # Nama variabel (mlp) dan urutan indeks (0, 3, 5) sudah DIJAMIN cocok dengan file .pt
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, 64), # Indeks 0
            nn.ReLU(),                    # Indeks 1
            nn.Dropout(0.2),              # Indeks 2
            nn.Linear(64, 32),            # Indeks 3
            nn.ReLU(),                    # Indeks 4
            nn.Linear(32, 1)              # Indeks 5
        )

    def forward(self, u_idx, i_idx):
        u_e = self.u_emb(u_idx)
        i_e = self.i_emb(i_idx)
        concat = torch.cat([u_e, i_e], dim=1)
        return self.mlp(concat).squeeze()

# =========================================================
# 4. NARRE (Neural Attentional Rating Regression)
# =========================================================
class NARRE(nn.Module):
    def __init__(self, vocab_size, embed_dim, pretrained_matrix, num_filters=100, kernel_size=3):
        super(NARRE, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_matrix))
        self.embedding.weight.requires_grad = True
        self.cnn = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size)
        self.attention_layer = nn.Sequential(nn.Linear(num_filters, 32), nn.ReLU(), nn.Linear(32, 1))
        self.prediction = nn.Linear(num_filters * 2, 1)
        self.dropout = nn.Dropout(0.2)

    def extract_and_attend(self, seqs):
        batch_size, num_reviews, num_words = seqs.shape
        flat_seqs = seqs.view(-1, num_words)
        emb = self.embedding(flat_seqs).permute(0, 2, 1)
        conv_out = F.relu(self.cnn(emb))
        pooled_out = F.max_pool1d(conv_out, conv_out.shape[2]).squeeze(2)
        review_features = pooled_out.view(batch_size, num_reviews, -1)
        attn_scores = self.attention_layer(review_features).squeeze(2)
        mask = (seqs.sum(dim=2) == 0)
        attn_scores.masked_fill_(mask, -1e9)
        attn_weights = F.softmax(attn_scores, dim=1).unsqueeze(2)
        attended_features = (review_features * attn_weights).sum(dim=1)
        return self.dropout(attended_features)

    def forward(self, u_seqs, i_seqs):
        u_feat = self.extract_and_attend(u_seqs)
        i_feat = self.extract_and_attend(i_seqs)
        concat_feat = torch.cat([u_feat, i_feat], dim=1)
        return self.prediction(concat_feat).squeeze()

# =========================================================
# 5. MPCN (Multi-Pointer Co-Attention Networks)
# =========================================================
class MPCN(nn.Module):
    def __init__(self, vocab_size, embed_dim, pretrained_matrix, num_filters=100, kernel_size=3):
        super(MPCN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_matrix))
        self.embedding.weight.requires_grad = True
        self.cnn = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size, padding=1)
        self.transform_u = nn.Linear(num_filters, num_filters)
        self.transform_i = nn.Linear(num_filters, num_filters)
        self.prediction = nn.Linear(num_filters * 2, 1)
        self.dropout = nn.Dropout(0.2)

    def get_review_representation(self, seqs):
        batch_size, num_reviews, max_words = seqs.shape
        flat_seqs = seqs.view(-1, max_words)
        emb = self.embedding(flat_seqs).permute(0, 2, 1)
        conv_out = F.relu(self.cnn(emb))
        pooled_out = F.max_pool1d(conv_out, conv_out.shape[2]).squeeze(2)
        return pooled_out.view(batch_size, num_reviews, -1)

    def forward(self, u_seqs, i_seqs):
        u_rep = self.get_review_representation(u_seqs)
        i_rep = self.get_review_representation(i_seqs)
        u_trans = self.transform_u(u_rep)
        i_trans = self.transform_i(i_rep)
        S = torch.bmm(u_trans, i_trans.transpose(1, 2))
        attn_u = F.softmax(S.max(dim=2)[0], dim=1).unsqueeze(2)
        attn_i = F.softmax(S.max(dim=1)[0], dim=1).unsqueeze(2)
        u_final = (u_rep * attn_u).sum(dim=1)
        i_final = (i_rep * attn_i).sum(dim=1)
        concat_feat = torch.cat([u_final, i_final], dim=1)
        concat_feat = self.dropout(concat_feat)
        return self.prediction(concat_feat).squeeze()

print("✅ SEMUA ARSITEKTUR BERHASIL DIMUAT DI MEMORI!")

--- MEMBANGUN ARSITEKTUR SEMUA MODEL UNTUK PRESENTASI ---
✅ SEMUA ARSITEKTUR BERHASIL DIMUAT DI MEMORI!


In [3]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("2. Mencari lokasi file secara otomatis di Kaggle...")

# ==========================================================
# DEFINISIKAN DEVICE DULU DI SINI
# ==========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Menggunakan komputasi: {device.type.upper()}")

# ==========================================================
# KODE PINTAR V2: Mencari file sampai ke folder terdalam
# ==========================================================
BASE_PATH = ""
for root, dirs, files in os.walk("/kaggle/input/"):
    if "word2idx.pkl" in files:
        BASE_PATH = root # Menyimpan folder tempat file ditemukan
        break

if BASE_PATH == "":
    print("❌ ERROR: File word2idx.pkl tidak ditemukan sama sekali!")
else:
    print(f"✅ Folder file berhasil ditemukan di: {BASE_PATH}")

    # Load Kamus
    with open(f"{BASE_PATH}/word2idx.pkl", "rb") as f:
        word2idx = pickle.load(f)

    # Load Matriks
    embedding_matrix = np.load(f"{BASE_PATH}/embedding_matrix.npy")

    print("\n3. Membangun ulang model dan memuat bobot (weights) dari backup...")
    # Inisialisasi model
    model_narre = NARRE(vocab_size=len(word2idx), embed_dim=300, pretrained_matrix=embedding_matrix).to(device)
    model_mpcn = MPCN(vocab_size=len(word2idx), embed_dim=300, pretrained_matrix=embedding_matrix).to(device)

    # Load bobot (weights)
    model_narre.load_state_dict(torch.load(f"{BASE_PATH}/model_narre.pt", map_location=device))
    model_mpcn.load_state_dict(torch.load(f"{BASE_PATH}/model_mpcn.pt", map_location=device))

    # WAJIB ubah ke mode evaluasi (.eval()) agar model tidak belajar lagi
    model_narre.eval()
    model_mpcn.eval()

    print("✅ Otak model NARRE dan MPCN sudah kembali pintar dan siap digunakan!")

2. Mencari lokasi file secara otomatis di Kaggle...
💻 Menggunakan komputasi: CUDA
✅ Folder file berhasil ditemukan di: /kaggle/input/datasets/muhammadaspazikra/dataset-skripsi

3. Membangun ulang model dan memuat bobot (weights) dari backup...
✅ Otak model NARRE dan MPCN sudah kembali pintar dan siap digunakan!


In [4]:
import pandas as pd

print("4. Menyiapkan Data Pengujian...")
test_df = pd.read_parquet(f"{BASE_PATH}/test_df.parquet")

# Ambil contoh 10 item acak
selected_items = test_df['parent_asin'].drop_duplicates().head(10).tolist()

# Ambil maksimal 5 user untuk setiap item
sample_users_df = (
    test_df[test_df['parent_asin'].isin(selected_items)]
    [['user_id', 'parent_asin', 'rating']]
    .groupby('parent_asin')
    .head(5)
    .reset_index(drop=True)
)
print(f"Total baris data yang akan ditebak: {len(sample_users_df)} baris")

# Trik pyarrow untuk memfilter file 5GB
test_users = sample_users_df['user_id'].unique().tolist()
print(f"Mengambil riwayat asli untuk {len(test_users)} user unik dari file 5GB...")

df_user_filtered = pd.read_parquet(
    f"{BASE_PATH}/user_reviews.parquet", 
    engine="pyarrow", 
    filters=[('user_id', 'in', test_users)] 
)
print("✅ Riwayat berhasil diambil tanpa membuat RAM laptop hang!")

4. Menyiapkan Data Pengujian...
Total baris data yang akan ditebak: 50 baris
Mengambil riwayat asli untuk 47 user unik dari file 5GB...
✅ Riwayat berhasil diambil tanpa membuat RAM laptop hang!


In [5]:
import re

print("5. Memproses Teks menjadi Matriks 3D (Tokenisasi)...")

# --- A. PROSES SISI USER ---
def split_to_list(doc):
    words = str(doc).split()
    return [' '.join(words[i:i+50]) for i in range(0, min(500, len(words)), 50)]

df_user_filtered['user_reviews_list'] = df_user_filtered['user_doc'].apply(split_to_list)

if 'user_reviews_list' in sample_users_df.columns:
    sample_users_df = sample_users_df.drop(columns=['user_reviews_list'])

sample_users_df = pd.merge(sample_users_df, df_user_filtered[['user_id', 'user_reviews_list']], on='user_id', how='left')
sample_users_df['user_reviews_list'] = sample_users_df['user_reviews_list'].apply(lambda x: x if isinstance(x, list) else [])


# --- B. FUNGSI TOKENISASI 3D ---
MAX_REVIEWS = 10
MAX_WORDS = 50
def review_list_to_sequence(reviews):
    if not isinstance(reviews, list): reviews = []
    arr = np.zeros((MAX_REVIEWS, MAX_WORDS), dtype=np.int32)
    for i, text in enumerate(reviews[:MAX_REVIEWS]):
        words = str(text).split()[:MAX_WORDS]
        indices = [word2idx.get(w, 1) for w in words]
        arr[i, :len(indices)] = indices
    return arr

print("Tokenisasi sisi User...")
sample_users_df['u_seqs'] = sample_users_df['user_reviews_list'].apply(review_list_to_sequence)


# --- C. PROSES SISI ITEM (SKENARIO A, B, C) ---
print("Memuat dan memproses data Sintetis T5 (Sisi Item)...")
def clean_text_for_glove(text):
    text = str(text).lower()
    return re.sub(r'[^a-z0-9\s]', '', text)

# Load data Item (Gunakan K=10 sebagai contoh)
df_k10 = pd.read_parquet(f"{BASE_PATH}/cold_items_synthetic_k10.parquet")
df_meta = pd.read_parquet(f"{BASE_PATH}/df_meta_cold.parquet")

# Filter hanya item yang sedang diuji agar tidak membebani RAM
test_items = sample_users_df['parent_asin'].unique().tolist()
df_k10 = df_k10[df_k10['parent_asin'].isin(test_items)]
df_meta = df_meta[df_meta['parent_asin'].isin(test_items)]

# Gabungkan Meta dan Synthetic
df_item_eval = pd.merge(df_k10, df_meta[['parent_asin', 'clean_metadata_glove']], on='parent_asin', how='inner')

# Bentuk List Skenario
df_item_eval['item_reviews_A'] = df_item_eval['clean_metadata_glove'].apply(lambda x: [x])
df_item_eval['item_reviews_B'] = df_item_eval['synthetic_reviews_list'].apply(lambda x: [clean_text_for_glove(t) for t in x])
df_item_eval['item_reviews_C'] = df_item_eval.apply(lambda row: [row['clean_metadata_glove']] + row['item_reviews_B'], axis=1)

print("Tokenisasi sisi Item...")
df_item_eval['i_seqs_A'] = df_item_eval['item_reviews_A'].apply(review_list_to_sequence)
df_item_eval['i_seqs_B'] = df_item_eval['item_reviews_B'].apply(review_list_to_sequence)
df_item_eval['i_seqs_C'] = df_item_eval['item_reviews_C'].apply(review_list_to_sequence)

# Gabungkan sisi Item ke tabel Pengujian Utama
sample_users_df = pd.merge(sample_users_df, df_item_eval[['parent_asin', 'i_seqs_A', 'i_seqs_B', 'i_seqs_C']], on='parent_asin', how='inner')


# --- D. FUNGSI PREDIKSI & EKSEKUSI ---
def predict_text_model_3d(model, user_col, item_col, df):
    model.eval()
    u_tensors = torch.tensor(np.stack(df[user_col].values), dtype=torch.long).to(device)
    i_tensors = torch.tensor(np.stack(df[item_col].values), dtype=torch.long).to(device)
    with torch.no_grad():
        preds = model(u_tensors, i_tensors)
    return preds.cpu().numpy().flatten()

print("\n🚀 6. MENEBAK RATING DENGAN NARRE & MPCN...")
result_final_df = sample_users_df[['user_id', 'parent_asin', 'rating']].copy()
result_final_df = result_final_df.rename(columns={'rating': 'actual_rating'})

print("- Prediksi NARRE...")
result_final_df['NARRE_A'] = predict_text_model_3d(model_narre, 'u_seqs', 'i_seqs_A', sample_users_df)
result_final_df['NARRE_B'] = predict_text_model_3d(model_narre, 'u_seqs', 'i_seqs_B', sample_users_df)
result_final_df['NARRE_C'] = predict_text_model_3d(model_narre, 'u_seqs', 'i_seqs_C', sample_users_df)

print("- Prediksi MPCN...")
result_final_df['MPCN_A'] = predict_text_model_3d(model_mpcn, 'u_seqs', 'i_seqs_A', sample_users_df)
result_final_df['MPCN_B'] = predict_text_model_3d(model_mpcn, 'u_seqs', 'i_seqs_B', sample_users_df)
result_final_df['MPCN_C'] = predict_text_model_3d(model_mpcn, 'u_seqs', 'i_seqs_C', sample_users_df)

# Rapikan angka
for col in ['NARRE_A', 'NARRE_B', 'NARRE_C', 'MPCN_A', 'MPCN_B', 'MPCN_C']:
    result_final_df[col] = result_final_df[col].astype(float).round(4)
    
print("\n✅ SELESAI! Ini hasil akhirnya:")
display(result_final_df.head(10))

5. Memproses Teks menjadi Matriks 3D (Tokenisasi)...
Tokenisasi sisi User...
Memuat dan memproses data Sintetis T5 (Sisi Item)...
Tokenisasi sisi Item...

🚀 6. MENEBAK RATING DENGAN NARRE & MPCN...
- Prediksi NARRE...
- Prediksi MPCN...

✅ SELESAI! Ini hasil akhirnya:


,user_id,parent_asin,actual_rating,NARRE_A,NARRE_B,NARRE_C,MPCN_A,MPCN_B,MPCN_C
0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B00852JEPW,4.0,3.6143,4.1224,4.0888,4.8380,4.5805,4.5855
1,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,B00D2J8558,3.0,2.8280,3.3236,3.2859,2.1079,3.2601,3.2356
2,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08TTVLTWT,3.0,3.4919,3.5827,3.5715,3.1781,3.6105,3.6017
3,AEINY4XOINMMJCK5GZ3M6MMHBN6A,B00B20M8CU,4.0,4.5415,5.0766,5.0255,3.2887,4.9382,4.9430
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,B000LY1UMC,4.0,4.0918,4.5254,4.4671,3.5896,4.6176,4.6726
5,AFSKPY37N3C43SOI5IEXEK5JSIYA,B009WZRCO6,4.0,3.8578,4.6107,4.5185,3.5839,4.6614,4.6405
6,AFGEM6BXCYHUILEOA3P2ZYBEF2TA,B0879L79L7,5.0,4.3722,4.7211,4.6702,3.9484,4.7663,4.7546
7,AHOTTCH2V6EADWONZ3GAVU5KUP3A,B07QC4WTNC,5.0,3.0011,3.2737,3.2395,2.1029,3.2760,3.2217
8,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B088RD3BBW,5.0,3.9657,4.4621,4.4211,3.8091,4.4704,4.4627
9,AFZUK3MTBIBEDQOPAK3OATUOUKLA,B07VZYRKJQ,4.0,4.6716,4.2763,4.3644,4.3147,4.4514,4.4589


In [6]:
import pandas as pd
import numpy as np
import math
import torch
from sklearn.metrics import mean_squared_error

print("============================================================")
print("📊 1. EVALUASI RATING (RMSE & MSE)")
print("============================================================")

metrics_list = []
# Daftar model yang ada di kolom tabel Anda
models_to_evaluate = ['DeepCoNN_A', 'DeepCoNN_B', 'DeepCoNN_C', 
                      'NARRE_A', 'NARRE_B', 'NARRE_C', 
                      'MPCN_A', 'MPCN_B', 'MPCN_C', 'MF', 'NCF']

for col in result_final_df.columns:
    if col in models_to_evaluate:
        # Hitung MSE (Mean Squared Error)
        mse = mean_squared_error(result_final_df['actual_rating'], result_final_df[col])
        # Hitung RMSE (Root Mean Squared Error)
        rmse = math.sqrt(mse)
        
        metrics_list.append({
            'Model/Skenario': col, 
            'RMSE': round(rmse, 4), 
            'MSE': round(mse, 4)
        })

df_rating_metrics = pd.DataFrame(metrics_list)
display(df_rating_metrics)


📊 1. EVALUASI RATING (RMSE & MSE)


,Model/Skenario,RMSE,MSE
0,NARRE_A,1.2701,1.6131
1,NARRE_B,1.2003,1.4407
2,NARRE_C,1.1978,1.4348
3,MPCN_A,1.6663,2.7767
4,MPCN_B,1.2953,1.6777
5,MPCN_C,1.2644,1.5987


In [7]:
import os
import random
import pickle
import gc
import pandas as pd
import numpy as np
from tqdm import tqdm
from IPython.display import display, FileLink

print("=" * 90)
print("🚀 SMART LOAD/MEMBUAT DF_RANKING FIXED (1 POSITIF VS 99 NEGATIF)")
print("=" * 90)

# =========================================================
# 1. CARI BASE_PATH DATASET SEKALI SAJA
# =========================================================
BASE_PATH = ""
for root, dirs, files in os.walk("/kaggle/input/"):
    if "word2idx.pkl" in files:
        BASE_PATH = root
        break

if BASE_PATH == "":
    raise FileNotFoundError("❌ word2idx.pkl tidak ditemukan di /kaggle/input/")

print(f"  -> BASE_PATH ditemukan: {BASE_PATH}")

# File ranking yang sudah Anda upload ke Dataset Skripsi
input_ranking_file = f"{BASE_PATH}/df_ranking_99.parquet"
# Lokasi fallback jika file perlu dibuat ulang
output_file = "/kaggle/working/df_ranking_99.parquet"

# =========================================================
# 💡 SMART LOAD: prioritaskan file yang sudah ada di Kaggle Input
# =========================================================
if os.path.exists(input_ranking_file):
    RANKING_PATH = input_ranking_file
    print(f"✅ File ranking ditemukan di dataset: {RANKING_PATH}")
    print("Membaca langsung dari Kaggle Input; tidak membuat ulang negative sampling...")
    df_ranking_new = pd.read_parquet(RANKING_PATH)

elif os.path.exists(output_file):
    RANKING_PATH = output_file
    print(f"✅ File ranking ditemukan di working: {RANKING_PATH}")
    print("Membaca langsung dari file untuk menghemat RAM dan waktu...")
    df_ranking_new = pd.read_parquet(RANKING_PATH)

else:
    print("⚠️ File df_ranking_99.parquet belum ditemukan di dataset maupun working.")
    print("Memulai proses pembuatan dari awal...")

    # =========================================================
    # 2. LOAD DATA TEST DAN TRAIN
    # =========================================================
    test_cold_df = pd.read_parquet(f"{BASE_PATH}/test_df.parquet")
    try:
        train_df = pd.read_parquet(f"{BASE_PATH}/train_df.parquet")
        user_interacted_items = train_df.groupby("user_id")["parent_asin"].apply(set).to_dict()
        del train_df # Bebaskan RAM
        gc.collect()
    except FileNotFoundError:
        user_interacted_items = {}

    # =========================================================
    # 3. AMBIL POOL COLD ITEMS YANG BENAR-BENAR PUNYA REPRESENTASI
    # =========================================================
    meta_files = [f for f in os.listdir(BASE_PATH) if "meta" in f.lower() and f.endswith(".parquet")]
    if len(meta_files) == 0:
        raise FileNotFoundError("❌ File metadata parquet tidak ditemukan.")

    df_meta = pd.read_parquet(f"{BASE_PATH}/{meta_files[0]}")
    cold_meta_items = set(df_meta["parent_asin"].unique())
    del df_meta # Bebaskan RAM
    gc.collect()

    synthetic_item_sets = []
    for k in [5, 10, 20]:
        syn_path = f"{BASE_PATH}/cold_items_synthetic_k{k}.parquet"
        if os.path.exists(syn_path):
            df_syn = pd.read_parquet(syn_path, columns=["parent_asin"])
            synthetic_item_sets.append(set(df_syn["parent_asin"].unique()))

    test_items = set(test_cold_df["parent_asin"].unique())
    valid_cold_pool = test_items & cold_meta_items

    for s in synthetic_item_sets:
        valid_cold_pool = valid_cold_pool & s

    valid_cold_pool = list(valid_cold_pool)

    if len(valid_cold_pool) < 100:
        raise ValueError(f"❌ Cold item valid hanya {len(valid_cold_pool)}. Tidak cukup untuk 1 positif + 99 negatif.")

    test_cold_df = test_cold_df[test_cold_df["parent_asin"].isin(valid_cold_pool)].reset_index(drop=True)

    # =========================================================
    # 4. BUAT RANKING DATA
    # =========================================================
    random.seed(42)
    np.random.seed(42)
    N_NEG = 99
    ranking_rows = []
    skipped = 0
    valid_cold_pool_set = set(valid_cold_pool)

    print("  -> Membangun formasi 1 Positif vs 99 Negatif...")
    for query_num, row in tqdm(test_cold_df.iterrows(), total=len(test_cold_df)):
        user = row["user_id"]
        pos_item = row["parent_asin"]
        rating = row["rating"]
        interacted = user_interacted_items.get(user, set())

        candidates = list(valid_cold_pool_set - {pos_item} - interacted)
        if len(candidates) < N_NEG:
            skipped += 1
            continue

        negatives = random.sample(candidates, N_NEG)
        query_id = f"q_{query_num}"

        candidate_rows = [{
            "query_id": query_id, "user_id": user, "parent_asin": pos_item, "actual_rating": rating, "is_real": 1
        }]
        for neg_item in negatives:
            candidate_rows.append({
                "query_id": query_id, "user_id": user, "parent_asin": neg_item, "actual_rating": 0.0, "is_real": 0
            })

        random.shuffle(candidate_rows)
        ranking_rows.extend(candidate_rows)

    df_ranking_new = pd.DataFrame(ranking_rows)

    # =========================================================
    # 5. VALIDASI & SIMPAN
    # =========================================================
    query_size = df_ranking_new.groupby("query_id").size()
    positive_count = df_ranking_new.groupby("query_id")["is_real"].sum()

    assert query_size.min() == 100 and query_size.max() == 100, "❌ Ada query yang bukan 100 kandidat."
    assert positive_count.min() == 1 and positive_count.max() == 1, "❌ Ada query yang tidak punya tepat 1 item positif."

    df_ranking_new.to_parquet(output_file, engine="pyarrow", index=False)
    RANKING_PATH = output_file
    print(f"\n✅ Selesai! df_ranking_99.parquet berhasil dibuat dan disimpan di: {output_file}")


# =========================================================
# VALIDASI FILE YANG TER-LOAD
# =========================================================
required_cols = {"query_id", "user_id", "parent_asin", "actual_rating", "is_real"}
missing_cols = required_cols - set(df_ranking_new.columns)
if missing_cols:
    raise KeyError(f"❌ Kolom wajib pada df_ranking_99.parquet tidak lengkap: {sorted(missing_cols)}")

query_size = df_ranking_new.groupby("query_id").size()
positive_count = df_ranking_new.groupby("query_id")["is_real"].sum()

print(f"Total baris ter-load : {len(df_ranking_new):,}")
print(f"Total query unik     : {df_ranking_new['query_id'].nunique():,}")
print(f"Ukuran kandidat/query: {query_size.min()} s.d. {query_size.max()}")
print(f"Positif/query         : {positive_count.min()} s.d. {positive_count.max()}")
print(f"Path aktif            : {RANKING_PATH}")

assert query_size.min() == 100 and query_size.max() == 100, "❌ Ada query yang bukan 100 kandidat."
assert positive_count.min() == 1 and positive_count.max() == 1, "❌ Ada query yang tidak punya tepat 1 item positif."

# =========================================================
# MENAMPILKAN HASIL
# =========================================================
display(df_ranking_new.head(10))
if RANKING_PATH.startswith("/kaggle/working/"):
    display(FileLink(RANKING_PATH))

🚀 SMART LOAD/MEMBUAT DF_RANKING FIXED (1 POSITIF VS 99 NEGATIF)
⚠️ File df_ranking_99.parquet belum ditemukan. Memulai proses pembuatan dari awal...
  -> BASE_PATH ditemukan: /kaggle/input/datasets/muhammadaspazikra/dataset-skripsi
  -> Membangun formasi 1 Positif vs 99 Negatif...


100%|██████████| 441893/441893 [18:46<00:00, 392.14it/s]



✅ Selesai! df_ranking_99.parquet berhasil dibuat dan disimpan di: /kaggle/working/df_ranking_99.parquet


,query_id,user_id,parent_asin,actual_rating,is_real
0,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B00XHU87Z0,0.0,0
1,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B081R3C56F,0.0,0
2,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B013ZV8VLU,0.0,0
3,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B00RXF7BJ8,0.0,0
4,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B000WKY2MK,0.0,0
5,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B07W7B6BM4,0.0,0
6,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B017W13YIO,0.0,0
7,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B0BNHVJ8NX,0.0,0
8,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B07SV4YT4Y,0.0,0
9,q_0,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,B077HV15HL,0.0,0


/kaggle/working/df_ranking_99.parquet

In [8]:
[x for x in globals().keys() if "rank" in x.lower()]

['ranking_rows', 'df_ranking_new']

In [9]:
print(type(ranking_rows))
print(type(df_ranking_new))

print(df_ranking_new.shape)
print(df_ranking_new.columns.tolist())
print(df_ranking_new.head())

<class 'list'>
<class 'pandas.core.frame.DataFrame'>
(44189300, 5)
['query_id', 'user_id', 'parent_asin', 'actual_rating', 'is_real']
  query_id                       user_id parent_asin  actual_rating  is_real
0      q_0  AHITBJSS7KYUBVZPX7M2WJCOIVKQ  B00XHU87Z0            0.0        0
1      q_0  AHITBJSS7KYUBVZPX7M2WJCOIVKQ  B081R3C56F            0.0        0
2      q_0  AHITBJSS7KYUBVZPX7M2WJCOIVKQ  B013ZV8VLU            0.0        0
3      q_0  AHITBJSS7KYUBVZPX7M2WJCOIVKQ  B00RXF7BJ8            0.0        0
4      q_0  AHITBJSS7KYUBVZPX7M2WJCOIVKQ  B000WKY2MK            0.0        0


In [ ]:
import os
import gc
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import re
from sklearn.metrics import mean_squared_error
from IPython.display import display, FileLink, HTML

gc.collect()
torch.cuda.empty_cache()

print("=" * 120)
print("🚀 EVALUASI FINAL (RMSE, MSE, NDCG, HITRATE) + DIAGNOSTIK TIE SCORE")
print("=" * 120)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
final_results_all = []

# ============================================================
# 1. LOAD KAMUS, MATRIKS, DAN MEMBANGUN ARSITEKTUR
# ============================================================
BASE_PATH = ""
for root, dirs, files in os.walk("/kaggle/input/"):
    if "word2idx.pkl" in files:
        BASE_PATH = root 
        break

with open(f"{BASE_PATH}/user2idx.pkl", "rb") as f: user2idx = pickle.load(f)
with open(f"{BASE_PATH}/item2idx.pkl", "rb") as f: item2idx = pickle.load(f)
with open(f"{BASE_PATH}/word2idx.pkl", "rb") as f: word2idx = pickle.load(f)
embedding_matrix = np.load(f"{BASE_PATH}/embedding_matrix.npy")

num_users, num_items, vocab_size = len(user2idx), len(item2idx), len(word2idx)

class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64):
        super(MatrixFactorization, self).__init__()
        self.u_emb = nn.Embedding(num_users, embed_dim)
        self.i_emb = nn.Embedding(num_items, embed_dim)
        self.u_bias, self.i_bias = nn.Embedding(num_users, 1), nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
    def forward(self, u_idx, i_idx):
        return (self.u_emb(u_idx) * self.i_emb(i_idx)).sum(dim=1) + self.u_bias(u_idx).squeeze() + self.i_bias(i_idx).squeeze() + self.global_bias

class NCF(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64):
        super(NCF, self).__init__()
        self.u_emb, self.i_emb = nn.Embedding(num_users, embed_dim), nn.Embedding(num_items, embed_dim)
        self.mlp = nn.Sequential(nn.Linear(embed_dim * 2, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, u_idx, i_idx):
        return self.mlp(torch.cat([self.u_emb(u_idx), self.i_emb(i_idx)], dim=1)).squeeze()

class DeepCoNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, pretrained_matrix, num_filters=100, kernel_size=3, latent_dim=32):
        super(DeepCoNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_matrix))
        self.user_conv, self.item_conv = nn.Conv1d(embed_dim, num_filters, kernel_size), nn.Conv1d(embed_dim, num_filters, kernel_size)
        self.user_fc, self.item_fc = nn.Linear(num_filters, latent_dim), nn.Linear(num_filters, latent_dim)
        self.prediction = nn.Linear(latent_dim * 2, 1)
        self.dropout = nn.Dropout(0.2)
    def forward(self, u_seq, i_seq):
        u_emb, i_emb = self.embedding(u_seq).permute(0, 2, 1), self.embedding(i_seq).permute(0, 2, 1)
        u_feat = F.relu(self.user_fc(F.max_pool1d(F.relu(self.user_conv(u_emb)), u_emb.shape[2]-2).squeeze(2)))
        i_feat = F.relu(self.item_fc(F.max_pool1d(F.relu(self.item_conv(i_emb)), i_emb.shape[2]-2).squeeze(2)))
        return self.prediction(self.dropout(torch.cat([u_feat, i_feat], dim=1))).squeeze()

class NARRE(nn.Module):
    def __init__(self, vocab_size, embed_dim, pretrained_matrix, num_filters=100, kernel_size=3):
        super(NARRE, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_matrix))
        self.cnn = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size)
        self.attention_layer = nn.Sequential(nn.Linear(num_filters, 32), nn.ReLU(), nn.Linear(32, 1))
        self.prediction = nn.Linear(num_filters * 2, 1)
        self.dropout = nn.Dropout(0.2)
    def extract_and_attend(self, seqs):
        batch_size, num_reviews, num_words = seqs.shape
        conv_out = F.relu(self.cnn(self.embedding(seqs.view(-1, num_words)).permute(0, 2, 1)))
        review_features = F.max_pool1d(conv_out, conv_out.shape[2]).squeeze(2).view(batch_size, num_reviews, -1)
        attn_scores = self.attention_layer(review_features).squeeze(2).masked_fill_((seqs.sum(dim=2) == 0), -1e9)
        return self.dropout((review_features * F.softmax(attn_scores, dim=1).unsqueeze(2)).sum(dim=1))
    def forward(self, u_seqs, i_seqs):
        return self.prediction(torch.cat([self.extract_and_attend(u_seqs), self.extract_and_attend(i_seqs)], dim=1)).squeeze()

class MPCN(nn.Module):
    def __init__(self, vocab_size, embed_dim, pretrained_matrix, num_filters=100, kernel_size=3):
        super(MPCN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_matrix))
        self.cnn = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size, padding=1)
        self.transform_u, self.transform_i = nn.Linear(num_filters, num_filters), nn.Linear(num_filters, num_filters)
        self.prediction = nn.Linear(num_filters * 2, 1)
        self.dropout = nn.Dropout(0.2)
    def get_review_representation(self, seqs):
        batch_size, num_reviews, max_words = seqs.shape
        return F.max_pool1d(F.relu(self.cnn(self.embedding(seqs.view(-1, max_words)).permute(0, 2, 1))), max_words).squeeze(2).view(batch_size, num_reviews, -1)
    def forward(self, u_seqs, i_seqs):
        u_rep, i_rep = self.get_review_representation(u_seqs), self.get_review_representation(i_seqs)
        S = torch.bmm(self.transform_u(u_rep), self.transform_i(i_rep).transpose(1, 2))
        u_final = (u_rep * F.softmax(S.max(dim=2)[0], dim=1).unsqueeze(2)).sum(dim=1)
        i_final = (i_rep * F.softmax(S.max(dim=1)[0], dim=1).unsqueeze(2)).sum(dim=1)
        return self.prediction(self.dropout(torch.cat([u_final, i_final], dim=1))).squeeze()

model_mf = MatrixFactorization(num_users=num_users, num_items=num_items, embed_dim=64).to('cpu')
model_ncf = NCF(num_users=num_users, num_items=num_items, embed_dim=64).to('cpu')
model_deepconn = DeepCoNN(vocab_size=vocab_size, embed_dim=300, pretrained_matrix=embedding_matrix).to('cpu')
model_narre = NARRE(vocab_size=vocab_size, embed_dim=300, pretrained_matrix=embedding_matrix).to('cpu')
model_mpcn = MPCN(vocab_size=vocab_size, embed_dim=300, pretrained_matrix=embedding_matrix).to('cpu')

model_mf.load_state_dict(torch.load(f"{BASE_PATH}/model_mf.pt", map_location='cpu'))
model_ncf.load_state_dict(torch.load(f"{BASE_PATH}/model_ncf.pt", map_location='cpu'))
model_deepconn.load_state_dict(torch.load(f"{BASE_PATH}/model_deepconn.pt", map_location='cpu'))
model_narre.load_state_dict(torch.load(f"{BASE_PATH}/model_narre.pt", map_location='cpu'))
model_mpcn.load_state_dict(torch.load(f"{BASE_PATH}/model_mpcn.pt", map_location='cpu'))

for m in [model_mf, model_ncf, model_deepconn, model_narre, model_mpcn]: m.eval()

# ============================================================
# 2. FUNGSI PREDIKSI & METRIK (RATING + RANKING)
# ============================================================
def predict_id(model_obj, df):
    model_obj.eval().to(device)
    preds = []
    with torch.no_grad():
        for start in range(0, len(df), 4096):
            batch = df.iloc[start:start+4096]
            preds.append(model_obj(torch.tensor(batch["u_idx"].values, dtype=torch.long, device=device), 
                                   torch.tensor(batch["i_idx"].values, dtype=torch.long, device=device)).cpu().numpy().flatten())
    model_obj.to('cpu')
    return np.concatenate(preds)

def predict_2d(model_obj, user_col, item_col, df):
    model_obj.eval().to(device)
    preds = []
    with torch.no_grad():
        for start in range(0, len(df), 128):
            batch = df.iloc[start:start+128]
            preds.append(model_obj(torch.tensor(np.stack(batch[user_col].values), dtype=torch.long, device=device), 
                                   torch.tensor(np.stack(batch[item_col].values), dtype=torch.long, device=device)).cpu().numpy().flatten())
    model_obj.to('cpu')
    return np.concatenate(preds)

def predict_3d(model_obj, user_col, item_col, df):
    model_obj.eval().to(device)
    preds = []
    with torch.no_grad():
        for start in range(0, len(df), 16):
            batch = df.iloc[start:start+16]
            preds.append(model_obj(torch.tensor(np.stack(batch[user_col].values), dtype=torch.long, device=device), 
                                   torch.tensor(np.stack(batch[item_col].values), dtype=torch.long, device=device)).cpu().numpy().flatten())
    model_obj.to('cpu')
    return np.concatenate(preds)

def get_rating_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return round(math.sqrt(mse), 4), round(mse, 4)

# 💡 UPDATE: Fungsi get_ranking_metrics yang lebih aman
def get_ranking_metrics(df, score_col, k):
    ndcg, hr, valid_queries = 0.0, 0.0, 0
    group_col = "query_id" if "query_id" in df.columns else "user_id"

    for qid, group in df.groupby(group_col):
        if group["is_real"].sum() != 1:
            continue

        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        rank = group.index[group["is_real"] == 1][0]

        if rank < k:
            hr += 1.0
            ndcg += 1.0 / math.log2(rank + 2)

        valid_queries += 1

    if valid_queries == 0:
        return 0.0, 0.0

    return round(ndcg / valid_queries, 4), round(hr / valid_queries, 4)

# 💡 TAMBAHAN: Fungsi Cek Tie Score untuk Pembuktian Skripsi
def cek_tie_score(df, score_col="score", model_name=""):
    group_col = "query_id" if "query_id" in df.columns else "user_id"
    stat = df.groupby(group_col)[score_col].agg(["nunique", "std", "min", "max"])
    
    print(f"\n--- Diagnostik Tie-Score untuk {model_name} ---")
    print("Jumlah query (User):", len(stat))
    print("Query dengan semua score SAMA/KEMBAR:", (stat["nunique"] <= 1).sum())
    print("Persentase query Tie Total:", round((stat["nunique"] <= 1).mean() * 100, 2), "%")

def save_result(model, sken, k_val, rmse, mse, ndcg, hr):
    final_results_all.append({'Model': model, 'Skenario': sken, 'K_Value': k_val, 'RMSE': rmse, 'MSE': mse, 'NDCG': ndcg, 'HitRate': hr})

# Fungsi Teks
def clean_text_for_glove(text): return re.sub(r'[^a-z0-9\s]', '', str(text).lower())
def split_to_list(doc): return [' '.join(str(doc).split()[i:i+50]) for i in range(0, min(500, len(str(doc).split())), 50)]
# DeepCoNN tetap memakai fungsi tokenisasi 2D yang sama untuk A/B/C.
# max_words=500 dipertahankan agar hasil evaluasi tetap sejalan dengan pipeline final sebelumnya.
def text_to_seq(text, max_words=500):
    arr = np.zeros(max_words, dtype=np.int32)
    indices = [word2idx.get(w, 1) for w in str(text).split()[:max_words]]
    arr[:len(indices)] = indices
    return arr
def list_to_seq(reviews, max_reviews=10, max_words=50):
    if not isinstance(reviews, list): reviews = []
    arr = np.zeros((max_reviews, max_words), dtype=np.int32)
    for i, text in enumerate(reviews[:max_reviews]):
        indices = [word2idx.get(w, 1) for w in str(text).split()[:max_words]]
        arr[i, :len(indices)] = indices
    return arr
def robust_merge(df_left, df_right, on_col):
    overlap = [c for c in df_left.columns if c in df_right.columns and c != on_col]
    return pd.merge(df_left.drop(columns=overlap, errors='ignore'), df_right, on=on_col, how='inner')


# ============================================================
# 3. SIAPKAN DATA RATING (TEST) & RANKING (1v99)
# ============================================================
print("\n[1/4] Menyiapkan Dataset Evaluasi...")

# Prioritaskan file ranking yang sudah disimpan di Dataset Skripsi.
# Gunakan /kaggle/working hanya sebagai fallback.
RANKING_PATH = f"{BASE_PATH}/df_ranking_99.parquet"
if not os.path.exists(RANKING_PATH):
    RANKING_PATH = "/kaggle/working/df_ranking_99.parquet"

if not os.path.exists(RANKING_PATH):
    raise FileNotFoundError(
        "❌ df_ranking_99.parquet tidak ditemukan di BASE_PATH maupun /kaggle/working/. "
        "Jalankan cell SMART LOAD/MEMBUAT DF_RANKING terlebih dahulu."
    )

print(f"✅ Memuat ranking dari: {RANKING_PATH}")
df_rank_eval = pd.read_parquet(RANKING_PATH)
df_test = pd.read_parquet(f"{BASE_PATH}/test_df.parquet").rename(columns={'rating': 'actual_rating'})

sample_users = df_rank_eval['user_id'].unique()[:300] 
df_rank_eval = df_rank_eval[df_rank_eval['user_id'].isin(sample_users)].copy()
df_rating_eval = df_test[df_test['user_id'].isin(sample_users)].copy()

df_user_hist = pd.read_parquet(f"{BASE_PATH}/user_reviews.parquet", engine="pyarrow", filters=[('user_id', 'in', sample_users.tolist())])
if 'clean_text' in df_user_hist.columns: df_user_hist = df_user_hist.rename(columns={'clean_text': 'user_doc'})
if 'user_reviews_list' not in df_user_hist.columns: df_user_hist['user_reviews_list'] = df_user_hist['user_doc'].apply(split_to_list)
df_user_hist['user_doc'] = df_user_hist['user_doc'].fillna("")
df_user_hist['user_seq'] = df_user_hist['user_doc'].apply(text_to_seq)
df_user_hist['u_seqs'] = df_user_hist['user_reviews_list'].apply(lambda x: x if isinstance(x, list) else []).apply(list_to_seq)
df_user_hist["u_idx"] = df_user_hist["user_id"].map(lambda x: user2idx.get(x, 0))

df_rank_eval = robust_merge(df_rank_eval, df_user_hist, 'user_id')
df_rank_eval["i_idx"] = df_rank_eval["parent_asin"].map(lambda x: item2idx.get(x, 0)) 

df_rating_eval = robust_merge(df_rating_eval, df_user_hist, 'user_id')
df_rating_eval["i_idx"] = df_rating_eval["parent_asin"].map(lambda x: item2idx.get(x, 0))

del df_user_hist, df_test
gc.collect()


# ============================================================
# 4. EVALUASI ID-BASED
# ============================================================
print("\n[2/4] Menguji Model MF & NCF...")
for model_name, model_obj in [("MF", model_mf), ("NCF", model_ncf)]:
    pred_rating = predict_id(model_obj, df_rating_eval)
    pred_ranking = predict_id(model_obj, df_rank_eval)
    
    df_temp_rank = df_rank_eval.copy()
    df_temp_rank['score'] = pred_ranking
    
    rmse, mse = get_rating_metrics(df_rating_eval['actual_rating'], pred_rating)
    ndcg, hr = get_ranking_metrics(df_temp_rank, 'score', k=10)
    
    # 💡 UPDATE: Ubah label "-" menjadi "K=10"
    save_result(model_name, "ID-Based", "K=10", rmse, mse, ndcg, hr)
    
    # Cek Tie Score untuk ID-Based
    if model_name == "MF": cek_tie_score(df_temp_rank, "score", "MF (ID-Based)")
    
    del df_temp_rank


# ============================================================
# 5. EVALUASI SKENARIO A (METADATA) — FIX KOLOM METADATA
# ============================================================
print("\n[3/4] Menguji Skenario A (Metadata)...")

# PENTING:
# Jangan memilih kolom metadata memakai candidate_cols[0], karena urutan kolom
# dapat membuat yang terambil adalah clean_description/clean_features yang kosong.
# File dan kolom dipilih secara eksplisit agar Skenario A/C benar-benar memakai
# metadata gabungan yang sama dengan pipeline awal.
META_PATH = f"{BASE_PATH}/df_meta_cold.parquet"
META_COL = "clean_metadata_glove"  # nama lama; embedding aktual tetap FastText

if not os.path.exists(META_PATH):
    raise FileNotFoundError(f"File metadata tidak ditemukan: {META_PATH}")

df_meta_raw = pd.read_parquet(META_PATH)
required_meta_cols = {"parent_asin", META_COL}
missing_meta_cols = required_meta_cols - set(df_meta_raw.columns)
if missing_meta_cols:
    raise KeyError(
        f"Kolom metadata wajib tidak ditemukan: {sorted(missing_meta_cols)}. "
        f"Kolom tersedia: {df_meta_raw.columns.tolist()}"
    )

# Ambil kolom metadata yang benar secara eksplisit.
df_item_meta = (
    df_meta_raw[["parent_asin", META_COL]]
    .drop_duplicates(subset="parent_asin")
    .rename(columns={META_COL: "clean_metadata_glove"})
    .copy()
)

df_item_meta["clean_metadata_glove"] = (
    df_item_meta["clean_metadata_glove"]
    .fillna("")
    .astype(str)
    .str.strip()
    .apply(clean_text_for_glove)
)

# Fallback hanya untuk baris yang metadata gabungannya kosong.
# Fallback dibangun dari title + clean_features + clean_description bila tersedia.
blank_meta = df_item_meta["clean_metadata_glove"].eq("")
if blank_meta.any():
    fallback_cols = [
        c for c in ["title", "clean_features", "clean_description", "features", "description"]
        if c in df_meta_raw.columns
    ]
    if fallback_cols:
        def _to_plain_text(value):
            if isinstance(value, (list, tuple, np.ndarray)):
                return " ".join(map(str, value))
            return "" if pd.isna(value) else str(value)

        fallback_df = df_meta_raw[["parent_asin"] + fallback_cols].drop_duplicates("parent_asin").copy()
        fallback_text = fallback_df[fallback_cols].apply(
            lambda row: " ".join(_to_plain_text(v) for v in row.values),
            axis=1
        ).apply(clean_text_for_glove)
        fallback_map = dict(zip(fallback_df["parent_asin"], fallback_text))
        df_item_meta.loc[blank_meta, "clean_metadata_glove"] = (
            df_item_meta.loc[blank_meta, "parent_asin"].map(fallback_map).fillna("")
        )

empty_meta_count = int(df_item_meta["clean_metadata_glove"].eq("").sum())
print(f"✅ Metadata eksplisit digunakan dari: {META_PATH}")
print(f"✅ Kolom metadata utama: {META_COL}")
print(f"⚠️ Jumlah metadata yang masih kosong setelah fallback: {empty_meta_count:,}")

if len(df_item_meta) == 0 or df_item_meta["clean_metadata_glove"].ne("").sum() == 0:
    raise ValueError("Semua metadata kosong. Evaluasi dihentikan agar hasil A/C tidak menyesatkan.")

df_item_meta["item_seq_A"] = df_item_meta["clean_metadata_glove"].apply(text_to_seq)
df_item_meta["i_seqs_A"] = df_item_meta["clean_metadata_glove"].apply(lambda x: list_to_seq([x]))

# Simpan salinan untuk Skenario C sebelum objek lain dibersihkan.
df_meta_cold_temp = df_item_meta[["parent_asin", "clean_metadata_glove"]].copy()

df_rating_A = robust_merge(df_rating_eval, df_item_meta, "parent_asin")
df_rank_A = robust_merge(df_rank_eval, df_item_meta, "parent_asin")

for name, obj, is_3d in [("DeepCoNN", model_deepconn, False), ("NARRE", model_narre, True), ("MPCN", model_mpcn, True)]:
    df_temp_rate = df_rating_A.copy()
    df_temp_rank = df_rank_A.copy()
    
    pred_rating = predict_3d(obj, "u_seqs", "i_seqs_A", df_temp_rate) if is_3d else predict_2d(obj, "user_seq", "item_seq_A", df_temp_rate)
    pred_ranking = predict_3d(obj, "u_seqs", "i_seqs_A", df_temp_rank) if is_3d else predict_2d(obj, "user_seq", "item_seq_A", df_temp_rank)
    
    df_temp_rank['score'] = pred_ranking
    rmse, mse = get_rating_metrics(df_temp_rate['actual_rating'], pred_rating)
    ndcg, hr = get_ranking_metrics(df_temp_rank, 'score', k=10)
    
    # 💡 UPDATE: Ubah label "-" menjadi "K=10"
    save_result(name, "Skenario A (Metadata)", "K=10", rmse, mse, ndcg, hr)
    
    # 💡 DIAGNOSTIK TIE SCORE: Membuktikan bahwa metadata kosong bikin skor seri
    cek_tie_score(df_temp_rank, "score", f"{name} (Skenario A)")
    
    del df_temp_rate, df_temp_rank

del df_item_meta, df_rating_A, df_rank_A
gc.collect()


# ============================================================
# 6. EVALUASI SKENARIO B & C (T5 SYNTHETIC)
# ============================================================
print("\n[4/4] Menguji Skenario B & C (K=5, 10, 20)...")
# df_meta_cold_temp sudah dibuat dari kolom metadata eksplisit pada Skenario A.

for k_val in [5, 10, 20]:
    print(f"  -> Memproses T5 K={k_val}...")
    t5_path = f"{BASE_PATH}/cold_items_synthetic_k{k_val}.parquet"
    if not os.path.exists(t5_path): continue

    df_t5 = pd.read_parquet(t5_path)
    df_item_k = robust_merge(df_t5[['parent_asin', 'synthetic_reviews_text', 'synthetic_reviews_list']], df_meta_cold_temp, "parent_asin")
    
    df_item_k["clean_synthetic"] = df_item_k["synthetic_reviews_text"].fillna("").astype(str).apply(clean_text_for_glove)
    df_item_k["item_seq_B"] = df_item_k["clean_synthetic"].apply(text_to_seq)
    df_item_k["item_seq_C"] = (df_item_k["clean_metadata_glove"] + " " + df_item_k["clean_synthetic"]).apply(text_to_seq)

    # VALIDASI KRUSIAL: pastikan input DeepCoNN B dan C tidak identik akibat metadata kosong/salah kolom.
    seq_B = np.stack(df_item_k["item_seq_B"].values)
    seq_C = np.stack(df_item_k["item_seq_C"].values)
    same_BC_mask = np.all(seq_B == seq_C, axis=1)
    same_BC_ratio = float(same_BC_mask.mean()) if len(same_BC_mask) else 1.0
    print(f"     Diagnostik DeepCoNN B=C: {same_BC_ratio:.2%} item memiliki sequence identik")

    if same_BC_ratio >= 1.0:
        raise RuntimeError(
            f"Semua item_seq_B dan item_seq_C identik pada K={k_val}. "
            "Periksa kembali isi metadata sebelum menjalankan evaluasi."
        )
    
    df_item_k["item_reviews_B"] = df_item_k["synthetic_reviews_list"].apply(lambda x: [clean_text_for_glove(str(t)) for t in x] if isinstance(x, (list, tuple, np.ndarray)) else [])
    df_item_k["item_reviews_C"] = df_item_k.apply(lambda row: [row["clean_metadata_glove"]] + row["item_reviews_B"], axis=1)
    
    df_item_k["i_seqs_B"] = df_item_k["item_reviews_B"].apply(list_to_seq)
    df_item_k["i_seqs_C"] = df_item_k["item_reviews_C"].apply(list_to_seq)
    
    df_item_k = df_item_k[["parent_asin", "item_seq_B", "item_seq_C", "i_seqs_B", "i_seqs_C"]]
    
    df_rating_k = robust_merge(df_rating_eval, df_item_k, "parent_asin")
    df_rank_k = robust_merge(df_rank_eval, df_item_k, "parent_asin")
    
    for sken, seq_2d, seq_3d in [("Skenario B (T5 Only)", "item_seq_B", "i_seqs_B"), ("Skenario C (Meta + T5)", "item_seq_C", "i_seqs_C")]:
        for name, obj, is_3d in [("DeepCoNN", model_deepconn, False), ("NARRE", model_narre, True), ("MPCN", model_mpcn, True)]:
            
            df_temp_rate = df_rating_k.copy()
            df_temp_rank = df_rank_k.copy()
            
            pred_rating = predict_3d(obj, "u_seqs", seq_3d, df_temp_rate) if is_3d else predict_2d(obj, "user_seq", seq_2d, df_temp_rate)
            pred_ranking = predict_3d(obj, "u_seqs", seq_3d, df_temp_rank) if is_3d else predict_2d(obj, "user_seq", seq_2d, df_temp_rank)
            
            df_temp_rank['score'] = pred_ranking
            rmse, mse = get_rating_metrics(df_temp_rate['actual_rating'], pred_rating)
            ndcg, hr = get_ranking_metrics(df_temp_rank, 'score', k=k_val)
            
            save_result(name, sken, f"K={k_val}", rmse, mse, ndcg, hr)
            del df_temp_rate, df_temp_rank

del df_meta_cold_temp
gc.collect()


# ============================================================
# 7. TABEL EVALUASI FINAL 
# ============================================================
print("\n" + "="*120)
print("📊 TABEL EVALUASI LENGKAP (RATING & RANKING) SEMUA MODEL")
print("="*120)

df_final_all = pd.DataFrame(final_results_all)
df_final_all['Model_Cat'] = pd.Categorical(df_final_all['Model'], categories=['MF', 'NCF', 'DeepCoNN', 'NARRE', 'MPCN'], ordered=True)
df_final_all = df_final_all.sort_values(by=['Model_Cat', 'Skenario', 'K_Value']).drop(columns=['Model_Cat']).reset_index(drop=True)

display(df_final_all)
df_final_all.to_csv("/kaggle/working/Hasil_Evaluasi_Final_Lengkap.csv", index=False)
display(FileLink("Hasil_Evaluasi_Final_Lengkap.csv"))

In [ ]:
import pandas as pd
import numpy as np
import torch
from IPython.display import display, HTML, FileLink

print("=" * 120)
print("🔬 ANALISIS ABLASI MENDALAM: PERBANDINGAN PREDIKSI RATING SEMUA MODEL")
print("=" * 120)

try:
    # 1. Pilih 1 User Acak dan Ambil 5 Barang Kandidatnya (Campuran Asli & Palsu)
    # Asumsi df_rank_eval dan df_rating_eval masih ada di memori
    uid_sample = df_rank_eval['user_id'].iloc[0]
    df_sample = df_rank_eval[df_rank_eval['user_id'] == uid_sample].head(5).copy()
    
    print(f"👤 Memproses Inferensi untuk User ID: {uid_sample} (5 Barang Kandidat)...\n")

    # 2. Siapkan Teks Input untuk Skenario A, B, dan C (Gunakan T5 K=20)
    # FIX: gunakan file dan kolom metadata secara eksplisit.
    # Jangan memakai kandidat kolom pertama karena dapat mengambil clean_description yang kosong.
    META_PATH = f"{BASE_PATH}/df_meta_cold.parquet"
    META_COL = "clean_metadata_glove"  # nama lama, vektor aktual tetap FastText
    df_meta_raw = pd.read_parquet(META_PATH)

    if META_COL not in df_meta_raw.columns:
        raise KeyError(f"Kolom {META_COL} tidak ditemukan pada {META_PATH}")

    df_desc = df_meta_raw[['parent_asin', 'description']].drop_duplicates('parent_asin').copy()
    df_desc['description'] = df_desc['description'].apply(
        lambda x: ' '.join(map(str, x)) if isinstance(x, (list, tuple, np.ndarray)) else str(x)
    )

    df_meta_clean = (
        df_meta_raw[['parent_asin', META_COL]]
        .drop_duplicates('parent_asin')
        .rename(columns={META_COL: 'clean_metadata_glove'})
        .copy()
    )
    df_meta_clean['clean_metadata_glove'] = (
        df_meta_clean['clean_metadata_glove']
        .fillna("")
        .astype(str)
        .str.strip()
        .apply(clean_text_for_glove)
    )

    # Ambil T5 (K=20)
    df_t5 = pd.read_parquet(f"{BASE_PATH}/cold_items_synthetic_k20.parquet")
    df_t5 = df_t5[['parent_asin', 'synthetic_reviews_text', 'synthetic_reviews_list']]
    df_t5['synthetic_reviews_text'] = df_t5['synthetic_reviews_text'].fillna("").astype(str)

    # Gabungkan semua ke df_sample
    df_sample = pd.merge(df_sample, df_desc, on='parent_asin', how='left')
    df_sample = pd.merge(df_sample, df_meta_clean, on='parent_asin', how='left')
    df_sample = pd.merge(df_sample, df_t5, on='parent_asin', how='left')

    # Buat Sequences untuk Inferensi
    df_sample['item_seq_A'] = df_sample['clean_metadata_glove'].apply(text_to_seq)
    df_sample['i_seqs_A'] = df_sample['clean_metadata_glove'].apply(lambda x: list_to_seq([x]))
    
    df_sample['clean_synthetic'] = df_sample['synthetic_reviews_text'].apply(clean_text_for_glove)
    df_sample['item_seq_B'] = df_sample['clean_synthetic'].apply(text_to_seq)
    df_sample['item_seq_C'] = (df_sample['clean_metadata_glove'] + " " + df_sample['clean_synthetic']).apply(text_to_seq)

    # Diagnostik input DeepCoNN: B dan C harus berbeda bila metadata valid.
    df_sample['DeepCoNN_Input_BC_Identik'] = [
        np.array_equal(b, c)
        for b, c in zip(df_sample['item_seq_B'], df_sample['item_seq_C'])
    ]
    print(
        "🔎 Input DeepCoNN B=C:",
        f"{df_sample['DeepCoNN_Input_BC_Identik'].mean():.2%}",
        "dari sampel"
    )
    if df_sample['DeepCoNN_Input_BC_Identik'].all():
        raise RuntimeError(
            "Semua input DeepCoNN B dan C masih identik. "
            "Metadata kosong atau belum masuk ke sequence."
        )
    
    df_sample['item_reviews_B'] = df_sample['synthetic_reviews_list'].apply(lambda x: [clean_text_for_glove(str(t)) for t in x] if isinstance(x, (list, tuple, np.ndarray)) else [])
    df_sample['item_reviews_C'] = df_sample.apply(lambda row: [row['clean_metadata_glove']] + row['item_reviews_B'], axis=1)
    
    df_sample['i_seqs_B'] = df_sample['item_reviews_B'].apply(list_to_seq)
    df_sample['i_seqs_C'] = df_sample['item_reviews_C'].apply(list_to_seq)

    # 3. Lakukan Prediksi untuk Semua Model & Skenario
    df_hasil = df_sample[['parent_asin', 'description', 'synthetic_reviews_text', 'is_real', 'actual_rating']].copy()
    df_hasil = df_hasil.rename(columns={
        'parent_asin': 'ID_Barang',
        'description': 'Deskripsi_Asli',
        'synthetic_reviews_text': 'Teks_T5',
        'actual_rating': 'Rating_Asli'
    })
    
    df_hasil['Status'] = df_hasil['is_real'].apply(lambda x: "✅ Asli" if x==1 else "❌ Palsu")

    print("🤖 Mengeksekusi Inferensi Model Baseline (ID-Based)...")
    df_hasil['Pred_MF'] = np.round(predict_id(model_mf, df_sample), 2)
    df_hasil['Pred_NCF'] = np.round(predict_id(model_ncf, df_sample), 2)

    print("🤖 Mengeksekusi Inferensi Model Teks (Skenario A, B, C)...")
    for name, model_obj, is_3d in [("DeepCoNN", model_deepconn, False), ("NARRE", model_narre, True), ("MPCN", model_mpcn, True)]:
        
        # Skenario A (Metadata Only)
        pred_A = predict_3d(model_obj, "u_seqs", "i_seqs_A", df_sample) if is_3d else predict_2d(model_obj, "user_seq", "item_seq_A", df_sample)
        df_hasil[f'{name}_Skenario_A'] = np.round(pred_A, 2)
        
        # Skenario B (T5 Only)
        pred_B = predict_3d(model_obj, "u_seqs", "i_seqs_B", df_sample) if is_3d else predict_2d(model_obj, "user_seq", "item_seq_B", df_sample)
        df_hasil[f'{name}_Skenario_B'] = np.round(pred_B, 2)
        
        # Skenario C (Meta + T5)
        pred_C = predict_3d(model_obj, "u_seqs", "i_seqs_C", df_sample) if is_3d else predict_2d(model_obj, "user_seq", "item_seq_C", df_sample)
        df_hasil[f'{name}_Skenario_C'] = np.round(pred_C, 2)

    # Selisih prediksi DeepCoNN B dan C untuk verifikasi langsung.
    df_hasil['Selisih_DeepCoNN_B_C'] = (
        df_hasil['DeepCoNN_Skenario_B'] - df_hasil['DeepCoNN_Skenario_C']
    ).abs()

    # 4. Rapikan Tampilan Tabel
    # Susun ulang kolom agar enak dibaca
    cols_order = ['ID_Barang', 'Status', 'Rating_Asli', 'Pred_MF', 'Pred_NCF', 
                  'DeepCoNN_Skenario_A', 'DeepCoNN_Skenario_B', 'DeepCoNN_Skenario_C', 'Selisih_DeepCoNN_B_C',
                  'NARRE_Skenario_A', 'NARRE_Skenario_B', 'NARRE_Skenario_C',
                  'MPCN_Skenario_A', 'MPCN_Skenario_B', 'MPCN_Skenario_C',
                  'Deskripsi_Asli', 'Teks_T5']
    
    df_hasil = df_hasil[cols_order].drop(columns=['is_real'], errors='ignore')
    
    print("\n🎉 Selesai! Berikut adalah Tabel Perbandingan Prediksi (Studi Ablasi) Anda:")
    pd.set_option('display.max_columns', None)
    display(HTML(df_hasil.head(5).to_html(index=False, justify='center')))
    
    # Simpan agar bisa dipindah ke Word/Excel
    csv_path = "/kaggle/working/Tabel_Perbandingan_Ablasi_Rating.csv"
    df_hasil.to_csv(csv_path, index=False)
    print(f"\n✅ Data berhasil disimpan ke: {csv_path}")
    display(FileLink("Tabel_Perbandingan_Ablasi_Rating.csv"))

except Exception as e:
    print(f"❌ Terjadi kesalahan: {e}\nPastikan Anda menjalankan cell ini tanpa merestart kernel dari kodingan utama sebelumnya.")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from IPython.display import display, HTML, FileLink

print("=" * 120)
print("🔬 ANALISIS ABLASI MENDALAM: PERBANDINGAN PREDIKSI RATING (RAW / TIDAK DIBULATKAN)")
print("=" * 120)

try:
    # 1. Pilih 1 User Acak dan Ambil 5 Barang Kandidatnya
    uid_sample = df_rank_eval['user_id'].iloc[0]
    df_sample = df_rank_eval[df_rank_eval['user_id'] == uid_sample].head(5).copy()
    
    print(f"👤 Memproses Inferensi untuk User ID: {uid_sample} (5 Barang Kandidat)...\n")

    # 2. Siapkan Teks Input untuk Skenario A, B, dan C
    # FIX: gunakan file dan kolom metadata secara eksplisit.
    # Jangan memakai kandidat kolom pertama karena dapat mengambil clean_description yang kosong.
    META_PATH = f"{BASE_PATH}/df_meta_cold.parquet"
    META_COL = "clean_metadata_glove"  # nama lama, vektor aktual tetap FastText
    df_meta_raw = pd.read_parquet(META_PATH)

    if META_COL not in df_meta_raw.columns:
        raise KeyError(f"Kolom {META_COL} tidak ditemukan pada {META_PATH}")

    df_desc = df_meta_raw[['parent_asin', 'description']].drop_duplicates('parent_asin').copy()
    df_desc['description'] = df_desc['description'].apply(
        lambda x: ' '.join(map(str, x)) if isinstance(x, (list, tuple, np.ndarray)) else str(x)
    )

    df_meta_clean = (
        df_meta_raw[['parent_asin', META_COL]]
        .drop_duplicates('parent_asin')
        .rename(columns={META_COL: 'clean_metadata_glove'})
        .copy()
    )
    df_meta_clean['clean_metadata_glove'] = (
        df_meta_clean['clean_metadata_glove']
        .fillna("")
        .astype(str)
        .str.strip()
        .apply(clean_text_for_glove)
    )

    df_t5 = pd.read_parquet(f"{BASE_PATH}/cold_items_synthetic_k20.parquet")
    df_t5 = df_t5[['parent_asin', 'synthetic_reviews_text', 'synthetic_reviews_list']]
    df_t5['synthetic_reviews_text'] = df_t5['synthetic_reviews_text'].fillna("").astype(str)

    df_sample = pd.merge(df_sample, df_desc, on='parent_asin', how='left')
    df_sample = pd.merge(df_sample, df_meta_clean, on='parent_asin', how='left')
    df_sample = pd.merge(df_sample, df_t5, on='parent_asin', how='left')

    df_sample['item_seq_A'] = df_sample['clean_metadata_glove'].apply(text_to_seq)
    df_sample['i_seqs_A'] = df_sample['clean_metadata_glove'].apply(lambda x: list_to_seq([x]))
    
    df_sample['clean_synthetic'] = df_sample['synthetic_reviews_text'].apply(clean_text_for_glove)
    df_sample['item_seq_B'] = df_sample['clean_synthetic'].apply(text_to_seq)
    df_sample['item_seq_C'] = (df_sample['clean_metadata_glove'] + " " + df_sample['clean_synthetic']).apply(text_to_seq)

    # Diagnostik input DeepCoNN: B dan C harus berbeda bila metadata valid.
    df_sample['DeepCoNN_Input_BC_Identik'] = [
        np.array_equal(b, c)
        for b, c in zip(df_sample['item_seq_B'], df_sample['item_seq_C'])
    ]
    print(
        "🔎 Input DeepCoNN B=C:",
        f"{df_sample['DeepCoNN_Input_BC_Identik'].mean():.2%}",
        "dari sampel"
    )
    if df_sample['DeepCoNN_Input_BC_Identik'].all():
        raise RuntimeError(
            "Semua input DeepCoNN B dan C masih identik. "
            "Metadata kosong atau belum masuk ke sequence."
        )
    
    df_sample['item_reviews_B'] = df_sample['synthetic_reviews_list'].apply(lambda x: [clean_text_for_glove(str(t)) for t in x] if isinstance(x, (list, tuple, np.ndarray)) else [])
    df_sample['item_reviews_C'] = df_sample.apply(lambda row: [row['clean_metadata_glove']] + row['item_reviews_B'], axis=1)
    
    df_sample['i_seqs_B'] = df_sample['item_reviews_B'].apply(list_to_seq)
    df_sample['i_seqs_C'] = df_sample['item_reviews_C'].apply(list_to_seq)

    # 3. Lakukan Prediksi (TANPA PEMBULATAN)
    df_hasil = df_sample[['parent_asin', 'description', 'synthetic_reviews_text', 'is_real', 'actual_rating']].copy()
    df_hasil = df_hasil.rename(columns={
        'parent_asin': 'ID_Barang',
        'description': 'Deskripsi_Asli',
        'synthetic_reviews_text': 'Teks_T5',
        'actual_rating': 'Rating_Asli'
    })
    df_hasil['Status'] = df_hasil['is_real'].apply(lambda x: "✅ Asli" if x==1 else "❌ Palsu")

    print("🤖 Mengeksekusi Inferensi Model Baseline (ID-Based)...")
    df_hasil['Pred_MF'] = predict_id(model_mf, df_sample)
    df_hasil['Pred_NCF'] = predict_id(model_ncf, df_sample)

    print("🤖 Mengeksekusi Inferensi Model Teks (Skenario A, B, C)...")
    for name, model_obj, is_3d in [("DeepCoNN", model_deepconn, False), ("NARRE", model_narre, True), ("MPCN", model_mpcn, True)]:
        
        pred_A = predict_3d(model_obj, "u_seqs", "i_seqs_A", df_sample) if is_3d else predict_2d(model_obj, "user_seq", "item_seq_A", df_sample)
        df_hasil[f'{name}_Skenario_A'] = pred_A
        
        pred_B = predict_3d(model_obj, "u_seqs", "i_seqs_B", df_sample) if is_3d else predict_2d(model_obj, "user_seq", "item_seq_B", df_sample)
        df_hasil[f'{name}_Skenario_B'] = pred_B
        
        pred_C = predict_3d(model_obj, "u_seqs", "i_seqs_C", df_sample) if is_3d else predict_2d(model_obj, "user_seq", "item_seq_C", df_sample)
        df_hasil[f'{name}_Skenario_C'] = pred_C

    # Selisih prediksi DeepCoNN B dan C untuk verifikasi langsung.
    df_hasil['Selisih_DeepCoNN_B_C'] = (
        df_hasil['DeepCoNN_Skenario_B'] - df_hasil['DeepCoNN_Skenario_C']
    ).abs()

    # 4. Rapikan Tampilan Tabel
    cols_order = ['ID_Barang', 'Status', 'Rating_Asli', 'Pred_MF', 'Pred_NCF', 
                  'DeepCoNN_Skenario_A', 'DeepCoNN_Skenario_B', 'DeepCoNN_Skenario_C', 'Selisih_DeepCoNN_B_C',
                  'NARRE_Skenario_A', 'NARRE_Skenario_B', 'NARRE_Skenario_C',
                  'MPCN_Skenario_A', 'MPCN_Skenario_B', 'MPCN_Skenario_C',
                  'Deskripsi_Asli', 'Teks_T5']
    
    df_hasil = df_hasil[cols_order].drop(columns=['is_real'], errors='ignore')
    
    print("\n🎉 Selesai! Berikut adalah Tabel Perbandingan Prediksi RAW (Tidak Dibulatkan):")
    pd.set_option('display.max_columns', None)
    display(HTML(df_hasil.head(5).to_html(index=False, justify='center')))
    
    csv_path = "/kaggle/working/Tabel_Perbandingan_Ablasi_Rating_RAW.csv"
    df_hasil.to_csv(csv_path, index=False)
    print(f"\n✅ Data berhasil disimpan ke: {csv_path}")
    display(FileLink("Tabel_Perbandingan_Ablasi_Rating_RAW.csv"))

except Exception as e:
    print(f"❌ Terjadi kesalahan: {e}")

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 100)
print("✅ VALIDASI FINAL METADATA DAN INPUT DEEPCONN")
print("=" * 100)

META_PATH = f"{BASE_PATH}/df_meta_cold.parquet"
META_COL = "clean_metadata_glove"  # nama lama; embedding aktual tetap FastText

df_cek = pd.read_parquet(META_PATH)
print("File metadata:", META_PATH)
print("Kolom metadata yang dipakai:", META_COL)
print("Daftar kolom:", df_cek.columns.tolist())

if META_COL not in df_cek.columns:
    raise KeyError(f"Kolom {META_COL} tidak ditemukan")

meta_clean = df_cek[META_COL].fillna("").astype(str).str.strip()
print("Jumlah baris metadata:", len(df_cek))
print("Jumlah metadata kosong:", int(meta_clean.eq("").sum()))
print("Persentase metadata kosong:", f"{meta_clean.eq('').mean():.2%}")

display(df_cek[["parent_asin", META_COL]].head(5))

if 'df_sample' in globals() and {'item_seq_B', 'item_seq_C'}.issubset(df_sample.columns):
    same_mask = np.array([
        np.array_equal(b, c)
        for b, c in zip(df_sample['item_seq_B'], df_sample['item_seq_C'])
    ])
    print("\\nProporsi input DeepCoNN B dan C yang identik:", f"{same_mask.mean():.2%}")
    if same_mask.all():
        raise RuntimeError("Semua input B dan C identik; evaluasi tidak boleh diteruskan.")
    print("✅ Validasi lolos: setidaknya sebagian input B dan C benar-benar berbeda.")
else:
    print("\\nℹ️ Jalankan cell analisis prediksi terlebih dahulu untuk memeriksa item_seq_B vs item_seq_C.")
